In [ ]:
# ══════════════════════════════════════════════════════════════════════
# H13 — Speed-Optimized: Batched Gravity + Parallel BPR
# ══════════════════════════════════════════════════════════════════════
#
# SPEED OPTIMIZATIONS over H12:
#
# 1. BATCHED GRAVITY:
#    Process 10 instances simultaneously on GPU instead of 1 at a time.
#    Energy + gradient for (B, P, n) in one backward() call.
#    ~10× speedup on gravity phase (GPU parallelism fully utilized).
#
# 2. bf16 MIXED PRECISION:
#    Half-precision energy computation via torch.amp.autocast.
#    ~1.5× bandwidth speedup, free on A100/V100/T4.
#
# 3. PARALLEL BPR:
#    ThreadPoolExecutor dispatches BPR runs across CPU cores.
#    Numba njit releases GIL → true multi-threaded parallelism.
#    ~2-4× speedup on BPR phase depending on core count.
#
# 4. PRE-EXTRACTED WEIGHTS:
#    All BSDT weights (conf/mfls/qs) extracted on GPU before BPR.
#    BPR workers are pure CPU — no GPU contention.
#
# 5. NUMBA SAT CHECK:
#    Fast CPU-side satisfiability check — no torch needed in BPR workers.
#
# Expected total speedup: 3-5× over H12
#   H12: ~329s for α=3.8/n=500 → H13: ~80-100s
#   H12: ~600s+ for α=4.0/n=1000 → H13: ~150-200s
#
# Solver logic (Tree-Walk BPR + GravityV3) UNCHANGED from H12.
# This is a pure performance optimization — same algorithm, faster execution.
#
# Author: Odeyemi Olusegun Israel
# ══════════════════════════════════════════════════════════════════════
import torch, numpy as np, time, math, os
from numba import njit
from concurrent.futures import ThreadPoolExecutor, as_completed

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

N_WORKERS = min(os.cpu_count() or 2, 8)
GRAVITY_BATCH = 10  # instances per GPU batch

print(f'CPU cores: {os.cpu_count()}, BPR workers: {N_WORKERS}')


# ── Instance generator ────────────────────────────────────────────────
def generate_3sat_instance(n, m):
    vars_idx = torch.randint(0, n, (m, 3))
    signs = torch.randint(0, 2, (m, 3)) * 2 - 1
    return list(zip(vars_idx.tolist(), signs.tolist()))


# ══════════════════════════════════════════════════════════════════════
#  TREE-WALK BPR — Anti-Fragmentation Local Search (Numba)
#  UNCHANGED from H12
# ══════════════════════════════════════════════════════════════════════

@njit(cache=True)
def bpr_treewalk(clauses_v, clauses_s, assignment, weight,
                 max_flips=200000, T_init=0.5, T_min=0.01,
                 p_random=0.1, beta=0.3, n_restarts=4,
                 branch_patience=80):
    m = clauses_v.shape[0]
    n = assignment.shape[0]
    restart_interval = max_flips // n_restarts

    # ── Build flat var→clause adjacency (once) ──
    var_count = np.zeros(n, dtype=np.int32)
    for c in range(m):
        for j in range(3):
            var_count[clauses_v[c, j]] += 1
    var_off = np.zeros(n + 1, dtype=np.int32)
    for vi in range(n):
        var_off[vi + 1] = var_off[vi] + var_count[vi]
    var_adj = np.zeros(var_off[n], dtype=np.int32)
    var_sign = np.zeros(var_off[n], dtype=np.int32)
    fill = np.zeros(n, dtype=np.int32)
    for c in range(m):
        for j in range(3):
            vi = clauses_v[c, j]
            pos = var_off[vi] + fill[vi]
            var_adj[pos] = c
            var_sign[pos] = clauses_s[c, j]
            fill[vi] += 1

    # ── Clause weights (SAPS-style) ──
    clause_w = np.ones(m, dtype=np.float64)

    # ── Init clause satisfaction ──
    clause_sat = np.zeros(m, dtype=np.int32)
    for c in range(m):
        for j in range(3):
            vi = clauses_v[c, j]
            s_ = clauses_s[c, j]
            if (assignment[vi] == 1 and s_ == 1) or (assignment[vi] == 0 and s_ == -1):
                clause_sat[c] += 1

    # ── Incremental unsat list ──
    unsat_list = np.zeros(m, dtype=np.int32)
    unsat_pos = np.full(m, -1, dtype=np.int32)
    n_unsat = 0
    for c in range(m):
        if clause_sat[c] == 0:
            unsat_pos[c] = n_unsat
            unsat_list[n_unsat] = c
            n_unsat += 1

    best_n_unsat = n_unsat
    best_assign = assignment.copy()

    # ── Branch state ──
    last_flipped = -1
    branch_stale = 0
    branch_progress = 0
    in_branch = False

    recent_size = 8
    recent_flipped = np.full(recent_size, -1, dtype=np.int32)
    recent_idx = 0

    for flip in range(max_flips):
        if n_unsat == 0:
            return assignment, flip

        if n_unsat < best_n_unsat:
            best_n_unsat = n_unsat
            for i in range(n):
                best_assign[i] = assignment[i]

        # ── WARM RESTART ──
        if flip > 0 and flip % restart_interval == 0 and n_unsat > 0:
            for ui in range(n_unsat):
                clause_w[unsat_list[ui]] += 1.0
            for c in range(m):
                clause_w[c] *= 0.95
            for i in range(n):
                assignment[i] = best_assign[i]
            n_perturb = max(3, n // 50)
            for _ in range(n_perturb):
                vi = np.random.randint(n)
                if np.random.random() < weight[vi] * 0.3:
                    assignment[vi] = 1 - assignment[vi]
            n_unsat = 0
            for c in range(m):
                clause_sat[c] = 0
                for j in range(3):
                    vi = clauses_v[c, j]
                    s_ = clauses_s[c, j]
                    if (assignment[vi] == 1 and s_ == 1) or (assignment[vi] == 0 and s_ == -1):
                        clause_sat[c] += 1
                if clause_sat[c] == 0:
                    unsat_pos[c] = n_unsat
                    unsat_list[n_unsat] = c
                    n_unsat += 1
                else:
                    unsat_pos[c] = -1
            if n_unsat == 0:
                return assignment, flip
            in_branch = False
            last_flipped = -1
            branch_stale = 0
            for ri in range(recent_size):
                recent_flipped[ri] = -1

        # ── TREE-WALK CLAUSE SELECTION ──
        ci = -1

        if in_branch and branch_stale < branch_patience:
            best_neighbor_w = -1.0
            for ri in range(recent_size):
                rv = recent_flipped[ri]
                if rv < 0:
                    continue
                for idx in range(var_off[rv], var_off[rv + 1]):
                    cc = var_adj[idx]
                    if clause_sat[cc] == 0 and clause_w[cc] > best_neighbor_w:
                        best_neighbor_w = clause_w[cc]
                        ci = cc
            if ci < 0:
                in_branch = False

        if not in_branch or ci < 0:
            ci = unsat_list[np.random.randint(n_unsat)]
            in_branch = True
            branch_stale = 0
            branch_progress = 0
            for ri in range(recent_size):
                recent_flipped[ri] = -1
            recent_idx = 0

        # ── Temperature ──
        local_prog = (flip % restart_interval) / restart_interval
        T = T_min + (T_init - T_min) * 0.5 * (1.0 + np.cos(3.141592653589793 * local_prog))

        # ── VARIABLE SELECTION ──
        if np.random.random() < p_random:
            v_flip = clauses_v[ci, np.random.randint(3)]
        else:
            int_brks = np.zeros(3, dtype=np.int32)
            scores = np.zeros(3, dtype=np.float64)

            for j in range(3):
                v_cand = clauses_v[ci, j]
                i_brk = 0; w_brk = 0.0; w_make = 0.0

                for idx in range(var_off[v_cand], var_off[v_cand + 1]):
                    cc2 = var_adj[idx]
                    s_here = var_sign[idx]
                    satisfies = ((assignment[v_cand] == 1 and s_here == 1) or
                                 (assignment[v_cand] == 0 and s_here == -1))
                    if satisfies:
                        if clause_sat[cc2] == 1:
                            i_brk += 1
                            w_brk += clause_w[cc2]
                    else:
                        if clause_sat[cc2] == 0:
                            w_make += clause_w[cc2]

                int_brks[j] = i_brk
                delta = w_brk - w_make
                scores[j] = np.exp(-delta / (T + 1e-10)) * (1.0 + beta * weight[v_cand])

            zb_n = 0
            zb_opts = np.zeros(3, dtype=np.int32)
            for j in range(3):
                if int_brks[j] == 0:
                    zb_opts[zb_n] = j
                    zb_n += 1

            if zb_n > 0:
                v_flip = clauses_v[ci, zb_opts[np.random.randint(zb_n)]]
            else:
                total = scores[0] + scores[1] + scores[2]
                if total < 1e-30:
                    v_flip = clauses_v[ci, np.random.randint(3)]
                else:
                    r = np.random.random() * total
                    if r <= scores[0]:
                        v_flip = clauses_v[ci, 0]
                    elif r <= scores[0] + scores[1]:
                        v_flip = clauses_v[ci, 1]
                    else:
                        v_flip = clauses_v[ci, 2]

        # ── Execute flip ──
        old_n_unsat = n_unsat
        assignment[v_flip] = 1 - assignment[v_flip]

        for idx in range(var_off[v_flip], var_off[v_flip + 1]):
            cc2 = var_adj[idx]
            s_here = var_sign[idx]
            old_sat = clause_sat[cc2]
            now_satisfies = ((assignment[v_flip] == 1 and s_here == 1) or
                             (assignment[v_flip] == 0 and s_here == -1))
            if now_satisfies:
                clause_sat[cc2] += 1
            else:
                clause_sat[cc2] -= 1
            new_sat = clause_sat[cc2]
            if old_sat == 0 and new_sat > 0:
                pos = unsat_pos[cc2]
                last = unsat_list[n_unsat - 1]
                unsat_list[pos] = last
                unsat_pos[last] = pos
                unsat_pos[cc2] = -1
                n_unsat -= 1
            elif old_sat > 0 and new_sat == 0:
                unsat_list[n_unsat] = cc2
                unsat_pos[cc2] = n_unsat
                n_unsat += 1

        if n_unsat < old_n_unsat:
            branch_stale = 0
            branch_progress += (old_n_unsat - n_unsat)
        else:
            branch_stale += 1

        last_flipped = v_flip
        recent_flipped[recent_idx % recent_size] = v_flip
        recent_idx += 1

    return best_assign, max_flips


# ══════════════════════════════════════════════════════════════════════
#  NUMBA SAT CHECK — CPU-only, no torch needed
# ══════════════════════════════════════════════════════════════════════

@njit(cache=True)
def check_sat(clauses_v, clauses_s, assignment):
    """Check if assignment satisfies all clauses. Pure CPU."""
    m = clauses_v.shape[0]
    for c in range(m):
        sat = False
        for j in range(3):
            vi = clauses_v[c, j]
            si = clauses_s[c, j]
            if (assignment[vi] == 1 and si == 1) or (assignment[vi] == 0 and si == -1):
                sat = True
                break
        if not sat:
            return False
    return True


# ══════════════════════════════════════════════════════════════════════
#  BATCHED ENERGY — process B instances × P particles at once
# ══════════════════════════════════════════════════════════════════════

def _energy_batched(s, mu_val, vars_t, signs_t):
    """
    Batched energy for multiple instances.
    s:       (B, P, n) continuous assignments
    vars_t:  (B, m, 3) long — clause variable indices
    signs_t: (B, m, 3) float — clause literal signs
    returns: (B, P)  energy per particle per instance
    """
    B, P, n = s.shape
    m = vars_t.shape[1]

    # Compute product (1-lit_0)(1-lit_1)(1-lit_2) one factor at a time
    # to minimize peak memory (avoids materializing (B,P,m,3) tensor)
    prod_val = torch.ones(B, P, m, device=s.device, dtype=s.dtype)
    for j in range(3):
        # idx_j: (B, m) → (B, 1, m) → (B, P, m)
        idx_j = vars_t[:, :, j].unsqueeze(1).expand(B, P, m)
        gathered_j = torch.gather(s, 2, idx_j)  # (B, P, m)
        lit_j = gathered_j * signs_t[:, None, :, j]  # (B, P, m)
        prod_val = prod_val * (1.0 - lit_j)

    e_sat = (prod_val / 8.0).sum(dim=-1)  # (B, P)

    if mu_val > 0:
        return e_sat + mu_val * ((1.0 - s * s) ** 2).sum(dim=-1)
    return e_sat


# ══════════════════════════════════════════════════════════════════════
#  BATCHED GRAVITY FLOW — process GRAVITY_BATCH instances at once
# ══════════════════════════════════════════════════════════════════════

def batched_gravity_flow(all_instances, n, m, steps=4000, particles=1000,
                         lr=0.05, momentum_beta=0.9, mu_scale=0.1,
                         G_max=0.10, top_k_frac=0.1, gravity_start=0.2,
                         elite_repulsion=0.5, gravity_interval=20,
                         batch_size=None):
    """
    Run GravityV3 for ALL instances, batched on GPU.

    Parameters
    ----------
    all_instances : list of clause-lists (each is list of (vars, signs))
    n, m : variable/clause counts (same for all instances)
    batch_size : how many instances per GPU batch (default: GRAVITY_BATCH=10)

    Returns
    -------
    list of (particles, n) tensors — one per instance
    """
    if batch_size is None:
        batch_size = GRAVITY_BATCH

    N_INST = len(all_instances)
    all_results = [None] * N_INST
    gi = gravity_interval
    use_amp = (device.type == 'cuda')

    for batch_start in range(0, N_INST, batch_size):
        batch_end = min(batch_start + batch_size, N_INST)
        B = batch_end - batch_start
        batch_instances = all_instances[batch_start:batch_end]

        # ── Stack clause tensors ──
        vars_list = []
        signs_list = []
        for clauses in batch_instances:
            vs = [v for v, s in clauses]
            ss = [s for v, s in clauses]
            vars_list.append(vs)
            signs_list.append(ss)

        vars_t = torch.tensor(vars_list, dtype=torch.long, device=device)      # (B, m, 3)
        signs_t = torch.tensor(signs_list, dtype=torch.float32, device=device)  # (B, m, 3)

        # ── Initialize particles ──
        s = (torch.randn(B, particles, n, device=device) * 0.3).clamp_(-0.9, 0.9)
        vel = torch.zeros_like(s)

        grav_step = int(gravity_start * steps)
        delay_step = int(0.7 * steps)
        top_k = max(1, int(top_k_frac * particles))
        theta = None

        best_e = torch.full((B, particles), float('inf'), device=device)
        plateau_count = torch.zeros(B, particles, device=device)
        decay_arr = 1.0 / (1.0 + 0.002 * torch.arange(
            steps, device=device, dtype=torch.float32))

        cached_targets = [None] * B

        for step in range(steps):
            # ── μ schedule (delay70) ──
            if step < delay_step:
                mu_val = 0.0
            else:
                t_l = (step - delay_step) / (steps - delay_step)
                mu_val = mu_scale * 0.5 * (1.0 - math.cos(math.pi * t_l))

            s = s.detach().requires_grad_(True)

            # ── Batched energy with bf16 ──
            if use_amp:
                with torch.amp.autocast('cuda', dtype=torch.bfloat16):
                    e = _energy_batched(s, mu_val, vars_t, signs_t)
                    e_f32 = e.float()
            else:
                e_f32 = _energy_batched(s, mu_val, vars_t, signs_t)

            e_vals = e_f32.detach()  # (B, P)
            e_f32.sum().backward()
            g = s.grad.detach().clone()

            with torch.no_grad():
                decay = decay_arr[step]

                # Plateau detection
                improved = e_vals < best_e
                best_e = torch.where(improved, e_vals, best_e)
                plateau_count = torch.where(improved,
                    torch.zeros_like(plateau_count), plateau_count + 1)
                pm = (plateau_count >= 50).unsqueeze(2)  # (B, P, 1)

                # Gnorm-adaptive step
                gnorm = g.norm(dim=2, keepdim=True).clamp_(min=1e-10)
                dte = (lr * decay) / (1.0 + 0.05 * gnorm)
                dte = dte * torch.where(pm, torch.tensor(2.0, device=device),
                                             torch.tensor(1.0, device=device))

                # Damping + energy amplification
                if theta is None:
                    theta = float(e_vals.median()) + 1e-8
                damp = 1.0 / (1.0 + e_vals.unsqueeze(2) / theta)
                gam = (e_vals.clamp(min=0) / (e_vals + 1.0)).unsqueeze(2)

                # Momentum update
                vel = momentum_beta * vel - dte * damp * (1.0 + gam) * g

                # Noise with plateau boost
                ns_base = 0.03 * decay
                noise = torch.randn_like(s) * torch.where(pm, 4.0 * ns_base, ns_base)

                s = (s + vel + noise).clamp_(-1, 1)

                # ── Gravity (per-instance, every gi steps) ──
                if step >= grav_step and (step - grav_step) % gi == 0:
                    progress = (step - grav_step) / (steps - grav_step)
                    g_mag = G_max * progress * progress * gi

                    for b in range(B):
                        _, top_idx = e_vals[b].topk(top_k, largest=False)
                        elite = s[b, top_idx]
                        d = torch.cdist(s[b].unsqueeze(0), elite.unsqueeze(0))[0]
                        cached_targets[b] = elite[d.argmin(dim=1)]

                        if top_k > 1:
                            ed = torch.cdist(elite.unsqueeze(0), elite.unsqueeze(0))[0]
                            ed.fill_diagonal_(float('inf'))
                            nn_e = ed.argmin(dim=1)
                            push = elite - elite[nn_e]
                            pn = push.norm(dim=1, keepdim=True).clamp_(min=1e-6)
                            s[b, top_idx] += (elite_repulsion * g_mag) * (push / pn)

                        s[b].add_(g_mag * (cached_targets[b] - s[b]))

                elif step >= grav_step:
                    for b in range(B):
                        if cached_targets[b] is not None:
                            progress = (step - grav_step) / (steps - grav_step)
                            g_mag = G_max * progress * progress
                            s[b].add_(g_mag * (cached_targets[b] - s[b]))

                s.clamp_(-1, 1)
                if (step + 1) % 200 == 0:
                    theta = float(e_vals.median()) + 1e-8

        # ── Store results for this batch ──
        s_final = s.detach()
        for b in range(B):
            all_results[batch_start + b] = s_final[b]  # (P, n)

        # Free GPU memory between batches
        del s, vel, g, e_vals, vars_t, signs_t, best_e, plateau_count
        torch.cuda.empty_cache() if device.type == 'cuda' else None

    return all_results


# ══════════════════════════════════════════════════════════════════════
#  PER-INSTANCE HELPERS — weight extraction + particle ranking
# ══════════════════════════════════════════════════════════════════════

class InstanceHelper:
    """Lightweight per-instance helper for weight extraction + BPR prep."""

    def __init__(self, n, clauses):
        self.n = n
        self.m = len(clauses)
        vs_list = [vs for vs, ss in clauses]
        ss_list = [ss for vs, ss in clauses]
        self.vars_t = torch.tensor(vs_list, dtype=torch.long, device=device)
        self.signs_t = torch.tensor(ss_list, dtype=torch.float32, device=device)
        self.pos_mask = (self.signs_t > 0).long()
        self.clauses_v = np.array(vs_list, dtype=np.int32)
        self.clauses_s = np.array(ss_list, dtype=np.int32)

    def find_best_particles(self, s, k=3):
        with torch.no_grad():
            x = (s > 0).long()
            lit_ok = (x[:, self.vars_t] == self.pos_mask.unsqueeze(0))
            n_sat = lit_ok.any(dim=2).sum(dim=1)
            topk_sat, topk_idx = n_sat.topk(k, largest=True)
            viols = self.m - topk_sat
            return topk_idx.cpu().tolist(), viols.cpu().tolist()

    def get_confidence(self, s, idx):
        with torch.no_grad():
            return (1.0 - s[idx].abs()).cpu().numpy().astype(np.float64)

    def get_mfls(self, s, idx):
        sb = s[idx].clone().detach().requires_grad_(True)
        lit = sb[self.vars_t] * self.signs_t
        e_sat = (torch.prod(1.0 - lit, dim=-1) / 8.0).sum()
        grad = torch.autograd.grad(e_sat, sb)[0]
        g = grad.abs().cpu().numpy().astype(np.float64)
        mx = g.max()
        if mx < 1e-10:
            return np.full(self.n, 0.5, dtype=np.float64)
        return g / mx

    def get_quadsurf(self, s, idx):
        with torch.no_grad():
            sb = s[idx]
            lit = sb[self.vars_t] * self.signs_t
            clause_e = (1.0 - lit).prod(dim=1) / 8.0
            qs = torch.zeros(self.n, device=device)
            for j in range(3):
                qs.scatter_add_(0, self.vars_t[:, j], clause_e)
            qs = qs.cpu().numpy().astype(np.float64)
            mx = qs.max()
            if mx < 1e-10:
                return np.full(self.n, 0.5, dtype=np.float64)
            return qs / mx


# ══════════════════════════════════════════════════════════════════════
#  PARALLEL BPR WORKER
# ══════════════════════════════════════════════════════════════════════

def bpr_worker(clauses_v, clauses_s, x_np, weight, max_flips,
               T_init, T_min, p_random, beta, n_restarts, branch_patience):
    """
    Run one BPR solve attempt. Pure CPU — no GPU or torch needed.
    Returns (is_sat, flips_used).
    """
    sol, flips = bpr_treewalk(
        clauses_v, clauses_s, x_np, weight,
        max_flips=max_flips, T_init=T_init, T_min=T_min,
        p_random=p_random, beta=beta, n_restarts=n_restarts,
        branch_patience=branch_patience
    )
    is_sat = check_sat(clauses_v, clauses_s, sol)
    return is_sat, flips


# ══════════════════════════════════════════════════════════════════════
#  COMPILE + WARMUP
# ══════════════════════════════════════════════════════════════════════

# Warmup Numba (BPR + SAT check)
_wv = np.array([[0, 1, 2]], dtype=np.int32)
_ws = np.array([[1, -1, 1]], dtype=np.int32)
_wa = np.array([1, 0, 1], dtype=np.int32)
_ww = np.array([0.5, 0.3, 0.8], dtype=np.float64)
_ = bpr_treewalk(_wv, _ws, _wa, _ww, max_flips=10, n_restarts=2)
_ = check_sat(_wv, _ws, _wa)

print()
print('✓ H13 Speed-Optimized loaded')
print(f'  Device:  {device}')
if device.type == 'cuda':
    print(f'  GPU:     {torch.cuda.get_device_name()}')
    print(f'  VRAM:    {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')
print(f'  Workers: {N_WORKERS} threads for parallel BPR')
print(f'  Batch:   {GRAVITY_BATCH} instances per gravity batch')
print()
print('  Speed Optimizations:')
print('    ┌─ Batched gravity: 10 instances × GPU simultaneously')
print('    ├─ bf16 autocast: halved bandwidth for energy computation')
print('    ├─ Parallel BPR: ThreadPool across CPU cores')
print('    ├─ Pre-extracted weights: GPU→CPU before BPR dispatch')
print('    └─ Numba SAT check: pure CPU verification')
print()
print('  Solver (unchanged from H12):')
print('    ├─ Tree-Walk BPR: anti-fragmentation walk')
print('    ├─ GravityV3: momentum β=0.9 + plateau escape')
print('    └─ 3 modes (Conf/MFLS/QS) × top-3 particles')

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# H13 Experiment: Batched Gravity + Parallel BPR
# ══════════════════════════════════════════════════════════════════════
import matplotlib.pyplot as plt

ALPHAS   = [3.8, 4.0, 4.2]
NS       = [500, 750, 1000]
N_INST   = 50
PARTICLES = 1000
STEPS    = 4000
BETA     = 0.3
MAX_FLIPS = 200000
N_RESTARTS = 4
TOP_K_PARTICLES = 3
BRANCH_PATIENCE = 80

MODES = ['confidence', 'mfls', 'quadsurf']

# All baselines
h9b = {
    (3.8, 500): 94.0, (3.8, 750): 86.0, (3.8, 1000): 72.0,
    (4.0, 500): 44.0, (4.0, 750): 26.0, (4.0, 1000):  4.0,
    (4.2, 500):  8.0, (4.2, 750):  0.0, (4.2, 1000):  0.0,
}
h10b = {
    (3.8, 500): 98.0, (3.8, 750): 98.0, (3.8, 1000): 96.0,
    (4.0, 500): 72.0, (4.0, 750): 60.0, (4.0, 1000): 30.0,
    (4.2, 500): 10.0, (4.2, 750):  6.0, (4.2, 1000):  0.0,
}
h11b_ens = {
    (3.8, 500): 100.0, (3.8, 750): 100.0, (3.8, 1000): 100.0,
    (4.0, 500): 68.0,  (4.0, 750): 76.0,  (4.0, 1000): 52.0,
    (4.2, 500):  8.0,  (4.2, 750):  2.0,  (4.2, 1000):  2.0,
}

results = {m: {} for m in MODES}
ensemble_results = {}
s1_results = {}

print('=' * 120)
print('H13 — Speed-Optimized: Batched Gravity + Parallel BPR')
print(f'  Batched gravity ({GRAVITY_BATCH}/batch) | {N_WORKERS} BPR workers')
print(f'  Tree-Walk BPR | patience={BRANCH_PATIENCE} | {N_RESTARTS}×{MAX_FLIPS//N_RESTARTS//1000}K restarts')
print(f'  Top-{TOP_K_PARTICLES} particles × {len(MODES)} modes | GravityV3 momentum')
print('=' * 120)
print(f"  {'α':>5} | {'n':>5} | {'S1':>3} | {'Conf%':>6} | {'MFLS%':>6} | "
      f"{'QS%':>6} | {'Ens%':>6} | {'H11bE':>6} | {'H10b':>5} | {'ΔvsH11b':>7} | Time  | Speedup")
print('  ' + '-' * 110)

# H12 reference times (from initial run, will be updated with full results)
h12_times = {}  # to be filled as H12 finishes

for alpha in ALPHAS:
    for n_var in NS:
        m_cls = int(alpha * n_var)
        t0 = time.time()

        # ═══════════════════════════════════════════════════════════════
        # PHASE 1: Generate all instances upfront
        # ═══════════════════════════════════════════════════════════════
        print(f'    α={alpha}, n={n_var}: Generating {N_INST} instances...')
        all_instances = [generate_3sat_instance(n_var, m_cls) for _ in range(N_INST)]
        t_gen = time.time() - t0

        # ═══════════════════════════════════════════════════════════════
        # PHASE 2: BATCHED GRAVITY — all 50 at once (in batches of 10)
        # ═══════════════════════════════════════════════════════════════
        t_grav_start = time.time()
        print(f'    α={alpha}, n={n_var}: Batched gravity '
              f'({N_INST} instances in {(N_INST + GRAVITY_BATCH - 1) // GRAVITY_BATCH} batches)...')

        all_particles = batched_gravity_flow(
            all_instances, n_var, m_cls,
            steps=STEPS, particles=PARTICLES,
            batch_size=GRAVITY_BATCH
        )

        t_grav = time.time() - t_grav_start
        print(f'    → Gravity done: {t_grav:.0f}s '
              f'({t_grav / N_INST:.1f}s/inst effective)')

        # ═══════════════════════════════════════════════════════════════
        # PHASE 3: Create helpers + find top-K + pre-extract weights
        # ═══════════════════════════════════════════════════════════════
        t_prep_start = time.time()
        s1_count = 0
        bpr_tasks = []  # (inst_idx, rank, mode, clauses_v, clauses_s, x_np, weight)

        for inst in range(N_INST):
            eng = InstanceHelper(n_var, all_instances[inst])
            sf = all_particles[inst]  # (P, n)
            top_indices, top_viols = eng.find_best_particles(sf, k=TOP_K_PARTICLES)

            # Check Stage 1 (gravity-only) solves
            gravity_solved = False
            for pi, viols in zip(top_indices, top_viols):
                if viols == 0:
                    s1_count += 1
                    gravity_solved = True
                    break

            if gravity_solved:
                # Mark all modes as solved for this instance
                bpr_tasks.append((inst, -1, 'gravity_solved', None, None, None, None))
                continue

            # Pre-extract weights on GPU (sequential, fast)
            for rank, (pi, viols) in enumerate(zip(top_indices, top_viols)):
                if viols == 0:
                    bpr_tasks.append((inst, rank, 'gravity_solved', None, None, None, None))
                    break

                x_np = (sf[pi] > 0).cpu().numpy().astype(np.int32)
                w_conf = eng.get_confidence(sf, pi)
                w_mfls = eng.get_mfls(sf, pi)
                w_qs = eng.get_quadsurf(sf, pi)

                for md, w in [('confidence', w_conf), ('mfls', w_mfls), ('quadsurf', w_qs)]:
                    bpr_tasks.append((inst, rank, md, eng.clauses_v, eng.clauses_s,
                                     x_np.copy(), w))

        t_prep = time.time() - t_prep_start
        print(f'    → Prep done: {t_prep:.1f}s (S1={s1_count} gravity-only solves)')

        # ═══════════════════════════════════════════════════════════════
        # PHASE 4: PARALLEL BPR — dispatch across CPU threads
        # ═══════════════════════════════════════════════════════════════
        t_bpr_start = time.time()

        # Organize tasks by instance and rank
        inst_tasks = {}  # inst_idx → [(rank, mode, cv, cs, x, w), ...]
        for task in bpr_tasks:
            inst_idx, rank, md, cv, cs, x, w = task
            if inst_idx not in inst_tasks:
                inst_tasks[inst_idx] = []
            inst_tasks[inst_idx].append((rank, md, cv, cs, x, w))

        # Track results
        mode_ok = {m: 0 for m in MODES}
        mode_flips = {m: 0 for m in MODES}
        mode_tried = {m: 0 for m in MODES}
        ens_ok = 0

        # Process each instance — parallelize BPR modes within each particle rank
        for inst_idx in range(N_INST):
            tasks = inst_tasks.get(inst_idx, [])

            # Check if gravity already solved it
            grav_solved = any(md == 'gravity_solved' for _, md, _, _, _, _ in tasks)
            if grav_solved:
                for md in MODES:
                    mode_ok[md] += 1
                ens_ok += 1
                continue

            # Group tasks by rank
            rank_tasks = {}
            for rank, md, cv, cs, x, w in tasks:
                if md == 'gravity_solved':
                    continue
                if rank not in rank_tasks:
                    rank_tasks[rank] = []
                rank_tasks[rank].append((md, cv, cs, x, w))

            any_solved = False
            inst_mode_solved = {m: False for m in MODES}

            for rank in sorted(rank_tasks.keys()):
                if any_solved and rank > 0:
                    break  # skip lower-ranked particles if rank 0 found solution

                mode_jobs = rank_tasks[rank]

                # ── Parallel BPR: submit all 3 modes simultaneously ──
                futures = {}
                with ThreadPoolExecutor(max_workers=min(N_WORKERS, len(mode_jobs))) as pool:
                    for md, cv, cs, x, w in mode_jobs:
                        f = pool.submit(
                            bpr_worker, cv, cs, x, w,
                            MAX_FLIPS, 0.5, 0.01, 0.1, BETA,
                            N_RESTARTS, BRANCH_PATIENCE
                        )
                        futures[f] = md

                    for f in as_completed(futures):
                        md = futures[f]
                        is_sat, flips = f.result()
                        if rank == 0:
                            mode_flips[md] += flips
                            mode_tried[md] += 1
                        if is_sat:
                            if rank == 0:
                                mode_ok[md] += 1
                            inst_mode_solved[md] = True
                            any_solved = True

            if any_solved:
                ens_ok += 1
                # For rank 0 modes not explicitly tested at rank>0, keep rank 0 stats
            else:
                # None solved at any rank — already counted in mode_tried
                pass

            # Progress update
            if (inst_idx + 1) % 10 == 0:
                el = time.time() - t0
                c_ = mode_ok['confidence'] / (inst_idx + 1) * 100
                m_ = mode_ok['mfls'] / (inst_idx + 1) * 100
                q_ = mode_ok['quadsurf'] / (inst_idx + 1) * 100
                e_ = ens_ok / (inst_idx + 1) * 100
                t_rem = el / (inst_idx + 1) * (N_INST - inst_idx - 1)
                print(f'    α={alpha}, n={n_var}: {inst_idx+1}/{N_INST}'
                      f'  C={c_:.0f}% M={m_:.0f}% Q={q_:.0f}% E={e_:.0f}%'
                      f'  ({el:.0f}s, ~{el + t_rem:.0f}s total)')

        t_bpr = time.time() - t_bpr_start
        elapsed = time.time() - t0

        # ═══════════════════════════════════════════════════════════════
        # RECORD + PRINT
        # ═══════════════════════════════════════════════════════════════
        for md in MODES:
            pct = mode_ok[md] / N_INST * 100
            af = mode_flips[md] // max(mode_tried[md], 1)
            results[md][(alpha, n_var)] = {'pct': pct, 'flips': af}

        ens_pct = ens_ok / N_INST * 100
        ensemble_results[(alpha, n_var)] = ens_pct
        s1_results[(alpha, n_var)] = s1_count

        c_ = results['confidence'][(alpha, n_var)]['pct']
        m_ = results['mfls'][(alpha, n_var)]['pct']
        q_ = results['quadsurf'][(alpha, n_var)]['pct']

        h11e = h11b_ens[(alpha, n_var)]
        delta_ens = ens_pct - h11e
        ws_ref = h10b[(alpha, n_var)]

        tag = '★' if ens_pct >= 95 else ('▲' if delta_ens > 2 else
              ('≈' if abs(delta_ens) <= 2 else '▼'))

        # Timing breakdown
        time_str = f'{elapsed:.0f}s'
        phase_str = f'(G:{t_grav:.0f} P:{t_prep:.0f} B:{t_bpr:.0f})'

        print(f'  {alpha:5.1f} | {n_var:5d} | {s1_count:3d}  | '
              f'{c_:5.1f}% | {m_:5.1f}% | {q_:5.1f}% | {ens_pct:5.1f}% | '
              f'{h11e:5.1f}% | {ws_ref:4.0f}% | {delta_ens:+6.1f}% | '
              f'{time_str:>5} | {phase_str} {tag}')
    print('  ' + '-' * 110)


# ══════════════════════════════════════════════════════════════════════
#  Full Summary
# ══════════════════════════════════════════════════════════════════════
print('\n' + '=' * 120)
print('FULL COMPARISON — H9b → H10b → H11b → H13')
print('=' * 120)
print(f"  {'α':>5} | {'n':>5} | {'H9b':>5} | {'H10b':>5} | {'H11bE':>6} | "
      f"{'H13C':>5} | {'H13M':>5} | {'H13Q':>5} | {'H13 Ens':>8} | {'Δ':>6}")
print('  ' + '-' * 85)
for alpha in ALPHAS:
    for n_var in NS:
        c_ = results['confidence'][(alpha, n_var)]['pct']
        m_ = results['mfls'][(alpha, n_var)]['pct']
        q_ = results['quadsurf'][(alpha, n_var)]['pct']
        e_ = ensemble_results[(alpha, n_var)]
        h9 = h9b[(alpha, n_var)]
        h10 = h10b[(alpha, n_var)]
        h11e = h11b_ens[(alpha, n_var)]
        delta = e_ - h11e
        print(f'  {alpha:5.1f} | {n_var:5d} | {h9:4.0f}% | {h10:4.0f}% | '
              f'{h11e:5.1f}% | {c_:4.1f}% | {m_:4.1f}% | {q_:4.1f}% | '
              f'{e_:7.1f}% | {delta:+5.1f}%')
    print('  ' + '-' * 85)


# ══════════════════════════════════════════════════════════════════════
#  Charts
# ══════════════════════════════════════════════════════════════════════
print('\nGenerating charts...\n')

fig, axes = plt.subplots(1, 3, figsize=(22, 7))
w = 0.15
colors = ['#e74c3c', '#3498db', '#f39c12', '#2ecc71', '#9b59b6']
labels_c = ['H9b (baseline)', 'H10b (WalkSAT)', 'H11b (BPR ens)',
            'H13 best single', 'H13 ensemble']

for i, n_var in enumerate(NS):
    ax = axes[i]
    x = np.arange(len(ALPHAS))

    bars_data = [
        [h9b[(a, n_var)] for a in ALPHAS],
        [h10b[(a, n_var)] for a in ALPHAS],
        [h11b_ens[(a, n_var)] for a in ALPHAS],
        [max(results['confidence'][(a, n_var)]['pct'],
             results['mfls'][(a, n_var)]['pct'],
             results['quadsurf'][(a, n_var)]['pct']) for a in ALPHAS],
        [ensemble_results[(a, n_var)] for a in ALPHAS],
    ]

    for k, (label, vals) in enumerate(zip(labels_c, bars_data)):
        offset = (k - 2) * w
        ax.bar(x + offset, vals, w, label=label, color=colors[k], alpha=0.85)

    ax.set_xticks(list(x))
    ax.set_xticklabels([str(a) for a in ALPHAS])
    ax.set_xlabel('α (clause ratio)')
    ax.set_ylabel('Solve Rate %')
    ax.set_title(f'n = {n_var}')
    ax.set_ylim(0, 105)
    ax.legend(fontsize=7, loc='upper right')
    ax.grid(axis='y', alpha=0.3)

fig.suptitle('H13: Speed-Optimized — Batched Gravity + Parallel BPR\n'
             'Same solver as H12 | Batched 10/GPU + threaded BPR',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('h13_speed_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: h13_speed_results.png')

# ── Timing Breakdown ─────────────────────────────────────────────────
print('\n' + '=' * 80)
print('SPEED ANALYSIS')
print('=' * 80)
print(f'\nBatched gravity: {GRAVITY_BATCH} instances/batch → '
      f'{(N_INST + GRAVITY_BATCH - 1) // GRAVITY_BATCH} batches per (α,n) combo')
print(f'Parallel BPR: {N_WORKERS} worker threads')
print()
print('Flip counts (lower = more efficient):')
print(f"  {'α':>5} | {'n':>5} | {'Conf':>8} | {'MFLS':>8} | {'QS':>8}")
print('  ' + '-' * 45)
for alpha in ALPHAS:
    for n_var in NS:
        cf = results['confidence'][(alpha, n_var)]['flips']
        mf = results['mfls'][(alpha, n_var)]['flips']
        qf = results['quadsurf'][(alpha, n_var)]['flips']
        print(f'  {alpha:5.1f} | {n_var:5d} | {cf:8d} | {mf:8d} | {qf:8d}')
    print('  ' + '-' * 45)

# ── Stage 1 improvement ─────────────────────────────────────────────
print('\nStage 1 (gravity-only solves):')
for alpha in ALPHAS:
    for n_var in NS:
        s1 = s1_results[(alpha, n_var)]
        if s1 > 0:
            print(f'  α={alpha}, n={n_var}: {s1}/50 solved by gravity alone ★')